# NEeds update to include medclipsamv2

# Per-Muscle and Overall Average Metrics — All Algorithms

For each algorithm, reads its per-muscle result CSVs, computes the mean of every
metric across all scans, appends an `Overall_Mean` row, and saves a summary CSV.
The final cell combines all `Overall_Mean` rows into one comparison table.

In [ ]:
import pathlib
import re
import pandas as pd

EVAL_DIR    = pathlib.Path(r'C:\Projects\dissector\eval_notebooks')
SUMMARY_DIR = EVAL_DIR / 'summary_results'
SUMMARY_DIR.mkdir(exist_ok=True)

# ── Canonical muscle names ────────────────────────────────────────────────────
MUSCLE_ALIASES = {
    'r_gracilis':  'R_gracilis',
    'l_gracilis':  'L_gracilis',
    'r_sartorius': 'R_sartorius',
    'l_sartorius': 'L_sartorius',
    'r_sart':      'R_sartorius',
    'l_sart':      'L_sartorius',
}

# ── Canonical metric names ────────────────────────────────────────────────────
# ORDER MATTERS — more specific patterns must come before general ones.
# inter_slice_dice_* must be before dice, otherwise the bare `dice` pattern
# would match them first (they contain the substring "dice").
_METRIC_RE = [
    ('inter_slice_dice_pred', re.compile(r'inter_slice_dice_pred',          re.I)),
    ('inter_slice_dice_gt',   re.compile(r'inter_slice_dice_gt',            re.I)),
    # Anchored: only matches when "dice" (or "lower_dice") is the entire bare column
    ('dice',                  re.compile(r'^(?:lower_)?dice$',              re.I)),
    ('hausdorff',             re.compile(r'hausdorff',                      re.I)),
    ('jaccard',               re.compile(r'jaccard',                        re.I)),
    ('volume_similarity',     re.compile(r'volume_similarity',              re.I)),
    ('false_negative',        re.compile(r'false.?neg|falseNeg',            re.I)),
    ('false_positive',        re.compile(r'false.?pos|falsePo',             re.I)),
    ('bce',                   re.compile(r'\bbce\b|binary_cross_entropy',   re.I)),
    ('boundary_iou_3d',       re.compile(r'boundary_iou',                  re.I)),
]

# Strip muscle-name prefix from a column so the pattern matches cleanly
_MUSCLE_PREFIX_RE = re.compile(r'^[RrLl]_(?:gracilis|sartorius|sart)_', re.I)

def canonical_metric(col):
    bare = _MUSCLE_PREFIX_RE.sub('', col).rstrip(':')
    for name, pat in _METRIC_RE:
        if pat.search(bare):
            return name
    return None


def extract_muscle(stem):
    s = stem.lower()
    for alias in sorted(MUSCLE_ALIASES, key=len, reverse=True):
        if f'_{alias}_' in s or s.endswith(f'_{alias}'):
            return MUSCLE_ALIASES[alias]
    return None


def process_algorithm(label, results_dir):
    """Return a summary DataFrame for one algorithm, or None if no CSVs found."""
    results_dir = pathlib.Path(results_dir)
    csv_files   = sorted(results_dir.glob('*.csv'))
    if not csv_files:
        print(f'  [skip] no CSVs in {results_dir}')
        return None

    rows = []
    for csv_path in csv_files:
        muscle = extract_muscle(csv_path.stem)
        if muscle is None:
            print(f'  [skip] could not identify muscle in {csv_path.name}')
            continue

        df = pd.read_csv(csv_path, index_col=0)
        metric_vals = {}
        for col in df.columns:
            m = canonical_metric(col)
            if m:
                metric_vals[m] = pd.to_numeric(df[col], errors='coerce').mean()

        row = {'muscle': muscle}
        row.update(metric_vals)
        rows.append(row)

    if not rows:
        return None

    summary = pd.DataFrame(rows).set_index('muscle')
    summary.insert(0, 'algorithm', label)

    overall = summary.select_dtypes(include='number').mean().rename('Overall_Mean')
    overall['algorithm'] = label
    summary = pd.concat([summary, overall.to_frame().T])
    summary.index.name = 'muscle'
    return summary


print('Helpers ready.')

In [ ]:
# ── Algorithm registry ────────────────────────────────────────────────────────
# Each entry: (display_label, results_dir_relative_to_EVAL_DIR)
# Add or remove entries here as new results become available.

REGISTRY = [
    # ── MuscleMap Thigh ───────────────────────────────────────────────────
    ('MuscleMap Thigh (water)',           'muscle_map_thigh/results_water'),
    ('MuscleMap Thigh (fat fraction)',    'muscle_map_thigh/results_fat_frac'),

    # ── MuscleMap Whole-Body ──────────────────────────────────────────────
    ('MuscleMap WB (water)',              'muscle_map_wb/results_water'),
    ('MuscleMap WB (fat fraction)',       'muscle_map_wb/results_fat_frac'),

    # ── MuscleMap WB + MedSAM bounding box ───────────────────────────────
    ('MM WB + MedSAM bbox (water)',       'muscle_map_wb_boxes_medsam/results_water'),
    ('MM WB + MedSAM bbox (fat fraction)','muscle_map_wb_boxes_medsam/results_fatfrac'),

    # ── MuscleMap WB + MedSAM logit mask ─────────────────────────────────
    ('MM WB + MedSAM logitmask (water)',        'muscle_map_wb_masks_medsam/results_water'),
    ('MM WB + MedSAM logitmask (fat fraction)', 'muscle_map_wb_masks_medsam/results_fat_frac'),

    # ── MuscleMap WB + SLM-SAM2 ──────────────────────────────────────────
    ('MM WB + SLM-SAM2 (water)',          'muscle_map_wb+slmsam/results_water'),
    ('MM WB + SLM-SAM2 (fat fraction)',   'muscle_map_wb+slmsam/results_fat_frac'),

    # ── Dafne ─────────────────────────────────────────────────────────────
    ('Dafne (water)',                     'dafne/results_water'),
    ('Dafne (fat fraction)',              'dafne/results_fat_frac'),

    # ── Dafne + MedSAM ────────────────────────────────────────────────────
    ('Dafne + MedSAM (water)',            'dafne_and_medsam/results_water'),
    ('Dafne + MedSAM (fat fraction)',     'dafne_and_medsam/results_fat_frac'),

    # ── Hirriririir / Multimodal Thigh ────────────────────────────────────
    ('Hirriririir (water)',               'multimodal-multiethnic/results_water'),
    ('Hirriririir (fat fraction)',        'multimodal-multiethnic/results_fat_frac'),

    # ── MuSeg ─────────────────────────────────────────────────────────────
    ('MuSeg (water)',                     'museg/results_water'),
    ('MuSeg (fat fraction)',              'museg/results_fat_frac'),
    ('MuSeg (dixon)',                     'museg/results_dixon'),

    # ── MedCLIP-SAMv2 ─────────────────────────────────────────────────────
    ('MedCLIP-SAMv2 (water)',             'medclipsamv2/results_water'),
    ('MedCLIP-SAMv2 (fat fraction)',      'medclipsamv2/results_fat_frac'),
]

print(f'{len(REGISTRY)} entries in registry.')
for label, rdir in REGISTRY:
    path = EVAL_DIR / rdir
    n = len(list(path.glob('*.csv'))) if path.exists() else 0
    status = f'{n} CSVs' if path.exists() else 'DIR MISSING'
    print(f'  {label}: {status}')

In [ ]:
# ── Process all algorithms ────────────────────────────────────────────────────
summaries = {}  # label -> DataFrame

for label, rdir in REGISTRY:
    print(f'\n── {label} ──')
    df = process_algorithm(label, EVAL_DIR / rdir)
    if df is None:
        continue
    summaries[label] = df

    # Display
    num_cols = df.select_dtypes(include='number').columns
    display(df.reset_index().style.format('{:.4f}', subset=num_cols).hide(axis='index'))

    # Save individual summary CSV
    safe_name = re.sub(r'[^\w]+', '_', label).strip('_').lower()
    out_path  = SUMMARY_DIR / f'{safe_name}_avg_metrics.csv'
    df.to_csv(out_path, float_format='%.4f')
    print(f'  Saved -> {out_path}')

print(f'\nProcessed {len(summaries)}/{len(REGISTRY)} algorithms.')

In [ ]:
# ── Combined Overall Means ────────────────────────────────────────────────────
overall_rows = [
    df.loc[['Overall_Mean']]
    for df in summaries.values()
    if 'Overall_Mean' in df.index
]

combined = pd.concat(overall_rows)
combined.index = [row['algorithm'] for _, row in combined.iterrows()]
combined.index.name = 'algorithm'
combined = combined.drop(columns='algorithm')

num_cols = combined.select_dtypes(include='number').columns
display(
    combined.reset_index()
    .style
    .format('{:.4f}', subset=num_cols)
    .hide(axis='index')
    .background_gradient(subset=['dice'], cmap='RdYlGn', axis=0)
    .background_gradient(subset=['hausdorff'], cmap='RdYlGn_r', axis=0)
)

out_combined = SUMMARY_DIR / 'overall_means.csv'
combined.to_csv(out_combined, float_format='%.4f')
print('Saved ->', out_combined)